# 00 — Análise do alvo: o gate empírico

Este notebook é **ponto de decisão bloqueante**. As trilhas de modelagem não
devem rodar antes de ele estar executado e com a seção de decisão preenchida.

Ele existe porque cinco escolhas do projeto foram feitas por hipótese e
precisavam de evidência antes de virarem compromisso.

| # | Pergunta | Decisão |
|---|---|---|
| 1 | Equipamentos é mesmo o melhor alvo? | D-01, D-18 |
| 2 | As chaves naturais declaradas são únicas? | semântica de `alterada` em `src/changes.py` |
| 3 | A densidade anual é suficiente? | D-04, D-10 |
| 4 | As coordenadas dão para a trilha geográfica? | D-15, D-17, D-22 |
| 5 | Que colunas o filtro empírico rejeita nos nove snapshots? | D-06 |

**Recorte:** este notebook roda sobre o **estado de São Paulo** (D-21). Rodou
antes sobre o município apenas, e a mudança de recorte alterou uma das
conclusões — ver o veredito 1. Trocar `RECORTE` abaixo recupera qualquer outro
recorte, porque ele é um prefixo de código IBGE.

Cada seção termina com um **veredito** escrito. Um número sem veredito não fecha
a decisão.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import changes, schema
from src.graph import RECORTE_PADRAO, filtro_recorte_sql
from src.paths import PRIMARY_FOLDER

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RECORTE = RECORTE_PADRAO          # '35' = estado de SP; '355030' = capital
FILTRO = filtro_recorte_sql(RECORTE)

PERIODOS = changes.periodos_disponiveis()
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")   # a máquina tem 9 GB no total (D-23)

print(f"Recorte: {RECORTE!r}  ->  {FILTRO}")
print(f"Snapshots na camada primária: {PERIODOS}")
print(f"Transições: {[str(t) for t in changes.transicoes(PERIODOS)]}")
print(f"Tabelas no escopo: {len(schema.FACT_TABLES)}")

# A checagem que faltava quando D-17 foi escrita com dados parciais: afirmar o
# número de snapshots antes de medir qualquer coisa. Ver D-22.
assert len(PERIODOS) >= 2, "rode `python -m src.pipeline` antes deste notebook"
if len(PERIODOS) < 9:
    print(f"\n[ATENÇÃO] apenas {len(PERIODOS)} de 9 snapshots convertidos. "
          "Qualquer número abaixo é parcial — foi exatamente assim que D-17 "
          "registrou um teto de cobertura errado.")

## 1. Qual tabela é o melhor alvo?

D-01 recomendou `rlEstabEquipamento` com base em duas competências e em contagens
**nacionais**. Aqui a comparação é feita sobre a série inteira e dentro do
recorte de fato, porque um rótulo pode ser denso no país e esparso na amostra.

Quatro candidatas, escolhidas por serem tabelas de fato ligadas ao
estabelecimento cujo conteúdo varia no tempo.

In [ ]:
CANDIDATAS = {
    "rlEstabEquipamento": "co_equipamento",
    "rlEstabComplementar": "co_leito",
    "rlEstabServClass": "co_servico",
    "rlEstabInstFisiAssist": "co_instalacao",
}

def cobertura(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        fato = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not fato.exists() or not raiz.exists():
            continue
        colunas = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{fato}')").fetchall()}
        if col_item not in colunas:
            linhas.append({"periodo": periodo, "erro": f"sem coluna {col_item}"})
            continue
        linhas.append(con.execute(f'''
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                WHERE {FILTRO}
            )
            SELECT '{periodo}' AS periodo,
                   (SELECT COUNT(*) FROM sel) AS estabelecimentos,
                   COUNT(*) AS linhas,
                   COUNT(DISTINCT f.co_unidade) AS com_registro,
                   COUNT(DISTINCT f."{col_item}") AS itens_distintos
            FROM read_parquet('{fato}') f JOIN sel USING (co_unidade)
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if "com_registro" in df:
        df["cobertura_%"] = (100 * df["com_registro"] / df["estabelecimentos"]).round(1)
    return df

for tabela, col in CANDIDATAS.items():
    print(f"\n{'=' * 78}\n{tabela}  (item = {col})\n{'=' * 78}")
    print(cobertura(tabela, col).to_string(index=False))

### O espaço de rótulos de cada candidata

Cobertura alta não basta. O que decide é quantos **eventos de aquisição** cada
candidata gera, porque é o que o modelo tem para aprender, e qual prevalência
resulta — muito baixa torna a tarefa estatisticamente frágil mesmo com muitos
eventos absolutos.

In [ ]:
def espaco_de_rotulos(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for t in changes.transicoes(PERIODOS):
        a = PRIMARY_FOLDER / t.origem / f"{tabela}.parquet"
        b = PRIMARY_FOLDER / t.destino / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / t.origem / "tbEstabelecimento.parquet"
        if not (a.exists() and b.exists() and raiz.exists()):
            continue
        linhas.append(con.execute(f'''
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                WHERE {FILTRO}
            ),
            itens AS (
                SELECT DISTINCT "{col_item}" FROM read_parquet('{b}')
                WHERE "{col_item}" IS NOT NULL
            ),
            tinha AS (
                SELECT DISTINCT co_unidade, "{col_item}"
                FROM read_parquet('{a}') JOIN sel USING (co_unidade)
            ),
            tem AS (
                SELECT DISTINCT co_unidade, "{col_item}"
                FROM read_parquet('{b}') JOIN sel USING (co_unidade)
            ),
            candidatos AS (
                SELECT s.co_unidade, i."{col_item}"
                FROM sel s CROSS JOIN itens i
                EXCEPT SELECT * FROM tinha
            )
            SELECT '{t.destino}' AS transicao,
                   (SELECT COUNT(*) FROM itens) AS itens,
                   COUNT(*) AS candidatos,
                   COUNT(m.co_unidade) AS aquisicoes
            FROM candidatos c
            LEFT JOIN tem m USING (co_unidade, "{col_item}")
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if "aquisicoes" in df:
        df["prevalencia_%"] = (100 * df["aquisicoes"] / df["candidatos"]).round(4)
    return df

resumos = {t: espaco_de_rotulos(t, c) for t, c in CANDIDATAS.items()}
for tabela, df in resumos.items():
    print(f"\n{'=' * 78}\n{tabela}\n{'=' * 78}")
    print(df.to_string(index=False))

In [ ]:
comparacao = pd.DataFrame([
    {
        "tabela": tabela,
        "itens": int(df["itens"].max()),
        "candidatos": int(df["candidatos"].sum()),
        "aquisicoes": int(df["aquisicoes"].sum()),
        "prevalencia_mediana_%": round(float(df["prevalencia_%"].median()), 4),
    }
    for tabela, df in resumos.items()
    if "aquisicoes" in df and not df.empty
]).sort_values("aquisicoes", ascending=False)

print(comparacao.to_string(index=False))

> **Veredito 1 — alvo. `rlEstabEquipamento` mantido, mas o argumento mudou.**
>
> No recorte estadual, somando as oito transições:
>
> | Tabela | Itens | Candidatos | Aquisições | Prevalência mediana |
> |---|---|---|---|---|
> | `rlEstabServClass` | 72 | 55.138.323 | **36.370** | 0,0512% |
> | `rlEstabEquipamento` | 99 | 73.446.373 | **34.571** | 0,0465% |
> | `rlEstabInstFisiAssist` | 51 | 36.881.306 | 19.914 | 0,0473% |
> | `rlEstabComplementar` | 69 | 53.687.388 | 2.718 | 0,0050% |
>
> **A expansão do recorte inverteu a liderança.** No município, equipamentos
> venciam com folga: 12.081 aquisições contra 8.992 de `rlEstabServClass`. No
> estado, serviços passam à frente por 5%. Isso não é ruído — é um efeito de
> composição: o estado tem proporcionalmente muito mais unidades pequenas, que
> registram serviço especializado com mais frequência do que adquirem
> equipamento.
>
> **Equipamentos permanece o alvo**, por três razões que não são o volume:
>
> 1. A pergunta de pesquisa é sobre **recurso físico**. Equipamento é um bem de
>    capital com custo, prazo de aquisição e sentido claro de escassez. Serviço
>    especializado é uma classificação administrativa, e "adquirir um serviço"
>    pode significar apenas reclassificar o que já se fazia.
> 2. Vocabulário maior — 99 tipos contra 72 — dá mais o que ranquear por
>    estabelecimento, que é onde a métrica de destaque mede (D-19).
> 3. A diferença de 5% em volume não compensa trocar um alvo interpretável por
>    um ambíguo.
>
> `rlEstabServClass` fica registrada como **alvo alternativo de primeira
> escolha**, agora com evidência de que sustentaria o experimento. Leitos
> continuam descartados: 2.718 eventos em oito transições, uma ordem de grandeza
> abaixo, e prevalência dez vezes menor.
>
> Registrado em D-18, revisado por D-21.

## 2. As chaves naturais declaradas são únicas?

`docs/01-selecao-tabelas.md` declara chave natural para `rlEstabEquipamento` e
`rlEstabComplementar` como **hipótese derivada do dicionário**.

Não é detalhe: sem chave única, `src/changes.py` não distingue modificação de
remoção seguida de inserção, e a taxa de mudança sai inflada — cada alteração
conta duas vezes.

In [ ]:
def unicidade(tabela: str) -> pd.DataFrame:
    chave = schema.CNES_NATURAL_KEY.get(tabela)
    if not chave:
        return pd.DataFrame([{"tabela": tabela, "obs": "sem chave natural declarada"}])
    cols = ", ".join(f'"{c}"' for c in chave)
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        if not p.exists():
            continue
        r = con.execute(f'''
            SELECT COUNT(*) AS linhas,
                   COUNT(*) - COUNT(DISTINCT ({cols})) AS duplicadas
            FROM read_parquet('{p}')
        ''').df().iloc[0]
        linhas.append({
            "tabela": tabela, "periodo": periodo, "chave": " + ".join(chave),
            "linhas": int(r["linhas"]), "duplicadas": int(r["duplicadas"]),
        })
    return pd.DataFrame(linhas)

for tabela in schema.CNES_NATURAL_KEY:
    print(unicidade(tabela).to_string(index=False), "\n")

In [ ]:
# Se houvesse duplicadas: que coluna a mais resolveria?
def chave_minima(tabela: str, periodo: str) -> pd.DataFrame:
    p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
    if not p.exists():
        return pd.DataFrame()
    base = list(schema.CNES_NATURAL_KEY.get(tabela, ()))
    extras = [c for c in schema.CNES_EXTRACT_COLUMNS[tabela]
              if c not in base and not c.startswith("to_char")]
    linhas = []
    for extra in [None, *extras]:
        cols = base + ([extra] if extra else [])
        if not cols:
            continue
        expr = ", ".join(f'"{c}"' for c in cols)
        dup = con.execute(f'''SELECT COUNT(*) - COUNT(DISTINCT ({expr}))
                              FROM read_parquet('{p}')''').fetchone()[0]
        linhas.append({"acrescentando": extra or "(chave declarada)",
                       "duplicadas": int(dup)})
    return pd.DataFrame(linhas).sort_values("duplicadas")

for tabela in schema.CNES_NATURAL_KEY:
    print(f"\n{tabela} em {PERIODOS[-1]}:")
    print(chave_minima(tabela, PERIODOS[-1]).to_string(index=False))

> **Veredito 2 — chaves naturais. As duas hipóteses estavam certas.**
>
> Zero duplicatas em todos os snapshots, para `rlEstabEquipamento` por
> (`co_unidade`, `co_equipamento`, `co_tipo_equipamento`, `tp_sus`) e
> `rlEstabComplementar` por (`co_unidade`, `co_leito`, `co_tipo_leito`).
>
> Deixam de ser hipóteses derivadas do dicionário e passam a fato verificado. A
> classificação `alterada` de `src/changes.py` é confiável para essas duas
> tabelas — e **só** para elas. As outras 42 caem no modo sem chave, em que cada
> modificação conta como remoção mais inserção.

## 3. A densidade anual é suficiente?

D-04 fixou nove snapshots anuais provisoriamente. D-10 pergunta se um ano de
intervalo esconde ciclos.

O sinal: taxa muito alta significa que o intervalo agrega eventos que se queria
separar. Taxa estável e moderada significa que o intervalo está adequado.

In [ ]:
try:
    taxa = changes.taxa_de_mudanca()
except FileNotFoundError as e:
    print(e)
    taxa = None

if taxa is not None:
    foco = taxa[taxa["tabela"].isin(CANDIDATAS)]
    print(foco[["tabela", "periodo_destino", "linhas_origem", "linhas_destino",
                "inserida", "removida", "alterada", "taxa_mudanca",
                "chave_declarada"]].to_string(index=False))

In [ ]:
if taxa is not None and not taxa.empty:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for tabela, grupo in taxa[taxa["tabela"].isin(CANDIDATAS)].groupby("tabela"):
        estilo = "-o" if grupo["chave_declarada"].all() else "--x"
        ax.plot(grupo["periodo_destino"], grupo["taxa_mudanca"], estilo, label=tabela)
    ax.set_title("Taxa de mudança anual por tabela candidata")
    ax.set_xlabel("transição (período de destino)")
    ax.set_ylabel("eventos / linhas")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Linha tracejada = sem chave natural declarada: cada modificação conta")
    print("como remoção mais inserção, então a taxa está superestimada.")

> **Veredito 3 — densidade de snapshots. Anual mantida.**
>
> Taxa de mudança de `rlEstabEquipamento` nas oito transições: 0,112, 0,094,
> 0,101, 0,100, 0,095, 0,085, 0,082, 0,089. Mediana **0,094**, amplitude inteira
> entre 0,082 e 0,112, sem pico nas transições de pandemia.
>
> Série plana é justamente o que indica que o intervalo não está agregando
> eventos que se queira separar. Não há evidência de ciclo escondido que
> justifique o custo de densificar. **D-10 fechada.**
>
> A mudança é dominada por inserção — de 63 mil a 89 mil por transição contra
> cerca de 10 mil remoções — coerente com o crescimento de 67% que motivou D-01.
>
> Leitura: `tbEstabelecimento` aparece com taxa acima de 1,0 e `alterada` zerada
> porque não tem chave natural declarada. É o comportamento documentado em
> `src/changes.py`, não defeito dos dados.

## 4. A trilha geográfica é viável?

A trilha 3 depende inteiramente de `nu_latitude` e `nu_longitude`.

**Esta seção tem história.** D-17 concluiu que o teto de cobertura era 57% e que
43% dos estabelecimentos jamais seriam nós. O número estava errado: a medição
rodou com seis das nove competências convertidas, e as três que faltavam eram as
de melhor cobertura. D-22 corrige. A célula de abertura deste notebook agora
afirma o número de snapshots justamente para que isso não se repita.

In [ ]:
def cobertura_geografica() -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not p.exists():
            continue
        linhas.append(con.execute(f'''
            SELECT '{periodo}' AS periodo,
                   COUNT(*) AS estabelecimentos,
                   COUNT(nu_latitude) AS com_coordenada,
                   SUM(CASE WHEN nu_latitude BETWEEN -34 AND 6
                             AND nu_longitude BETWEEN -74 AND -34
                             AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                            THEN 1 ELSE 0 END) AS plausivel
            FROM read_parquet('{p}') WHERE {FILTRO}
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    df["plausivel_%"] = (100 * df["plausivel"] / df["estabelecimentos"]).round(2)
    return df

geo = cobertura_geografica()
print(geo.to_string(index=False))

In [ ]:
# Cobertura acumulada: um estabelecimento posicionado em QUALQUER snapshot pode
# ser usado, porque D-15 trata a posição como invariante no tempo. É o teto real
# da trilha 3. `union_by_name` é obrigatório aqui — três tabelas têm colunas que
# somem e voltam entre competências (D-20).
arquivos = [str(PRIMARY_FOLDER / p / "tbEstabelecimento.parquet")
            for p in PERIODOS
            if (PRIMARY_FOLDER / p / "tbEstabelecimento.parquet").exists()]
lista = ", ".join(f"'{a}'" for a in arquivos)

acumulada = con.execute(f'''
    WITH sel AS (
        SELECT co_unidade, nu_latitude, nu_longitude
        FROM read_parquet([{lista}], union_by_name=true) WHERE {FILTRO}
    )
    SELECT COUNT(DISTINCT co_unidade) AS estabelecimentos,
           COUNT(DISTINCT CASE WHEN nu_latitude BETWEEN -34 AND 6
                                AND nu_longitude BETWEEN -74 AND -34
                                AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                               THEN co_unidade END) AS posicionaveis
    FROM sel
''').df()
acumulada["cobertura_acumulada_%"] = (
    100 * acumulada["posicionaveis"] / acumulada["estabelecimentos"]).round(2)
print(acumulada.to_string(index=False))
print(f"\nMelhor snapshot isolado: {geo['plausivel_%'].max():.2f}%")
print("Se a acumulada mal supera o melhor snapshot, o teto é estrutural: quem")
print("não tem coordenada hoje nunca teve. Foi o que se observou (D-17, D-22).")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(geo["periodo"], geo["plausivel_%"])
ax.axhline(float(acumulada["cobertura_acumulada_%"].iloc[0]), color="crimson",
           linestyle="--", label="acumulada (teto real)")
ax.set_title(f"Cobertura de coordenada plausível — recorte {RECORTE!r}")
ax.set_ylabel("% dos estabelecimentos")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

> **Veredito 4 — trilha geográfica. Viável, com 85,7% no estado.**
>
> | Snapshot | Município | Estado |
> |---|---|---|
> | 201701 | 0,5% | 1,1% |
> | 201901 | 3,8% | 26,4% |
> | 202001 | 33,8% | 71,7% |
> | 202201 | 57,6% | 79,9% |
> | 202501 | **74,7%** | **85,7%** |
>
> União das nove competências: **75,0%** no município, **85,7%** no estado.
>
> Há um degrau claro em 2020, coerente com o CNES ter passado a exigir
> geolocalização. A união acrescenta quase nada sobre o melhor snapshot isolado
> — 85,7% contra 85,7% —, então o teto é estrutural: quem não tem coordenada em
> 2025 nunca teve.
>
> **Correção registrada.** D-17 afirmou teto de 57% medindo seis de nove
> snapshots. D-22 corrige. A conclusão qualitativa sobreviveu; o nível não.
>
> **O que continua valendo.** D-15 trata a posição como invariante no tempo,
> tomada da observação mais antiga — não para elevar o teto, mas para dar nós à
> janela de treino de 2018 e 2019, onde a cobertura própria é de 1% a 26%.
> CEP-5 foi medido como substituto e rejeitado: raio mediano de 4,90 km contra
> 8,70 km de um controle embaralhado, informativo mas só duas vezes melhor que o
> azar. E o filtro de plausibilidade foi apertado, porque 1,2% das coordenadas
> caíam a até 197 km do centro numa cidade de 35 km.
>
> **Obrigação que permanece.** Comparar a trilha 3 com as outras duas exige o
> **mesmo subconjunto de nós**, senão a comparação mede diferença de amostra em
> vez de diferença de estrutura. `Previsao.mascara_de_entidades()` existe para
> isso.

## 5. O filtro empírico sobre os nove snapshots

D-06 aplicou o filtro usando apenas 201701 e 202501. Aqui ele é reaplicado sobre
a série inteira: uma coluna é degenerada se está 100% nula, ou constante, em
**todos** os snapshots.

Note o `union_by_name=true`. Sem ele a leitura falha — três das 44 tabelas têm
colunas que somem e voltam entre competências, com 201901 anômala (D-20).

In [ ]:
def triagem_empirica() -> pd.DataFrame:
    linhas = []
    for tabela, colunas in schema.CNES_EXTRACT_COLUMNS.items():
        arquivos = [PRIMARY_FOLDER / p / f"{tabela}.parquet" for p in PERIODOS]
        arquivos = [a for a in arquivos if a.exists()]
        if not arquivos:
            linhas.append({"tabela": tabela, "coluna": "*",
                           "motivo": "tabela ausente em todos os snapshots"})
            continue
        lista = ", ".join(f"'{a}'" for a in arquivos)
        fonte = f"read_parquet([{lista}], union_by_name=true)"
        presentes = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM {fonte}").fetchall()}
        total = con.execute(f"SELECT COUNT(*) FROM {fonte}").fetchone()[0]
        if total == 0:
            linhas.append({"tabela": tabela, "coluna": "*",
                           "motivo": "vazia em todos os snapshots"})
            continue

        usaveis = [c for c in colunas if c in presentes]
        for c in colunas:
            if c not in presentes:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "ausente do Parquet"})
        if not usaveis:
            continue
        # Uma query por tabela, não uma por coluna: 44 tabelas x ~9 colunas x 9
        # snapshots seriam milhares de varreduras dos mesmos arquivos.
        agregados = ", ".join(
            f'COUNT("{c}") AS nn{i}, COUNT(DISTINCT "{c}") AS nd{i}'
            for i, c in enumerate(usaveis))
        r = con.execute(f"SELECT {agregados} FROM {fonte}").df().iloc[0]
        for i, c in enumerate(usaveis):
            if r[f"nn{i}"] == 0:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "100% nula em todos os snapshots"})
            elif r[f"nd{i}"] <= 1:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "constante em todos os snapshots"})
    return pd.DataFrame(linhas)

rejeitadas = triagem_empirica()
uteis = sum(len(v) for v in schema.CNES_USEFUL_COLUMNS.values())
print(f"{len(rejeitadas)} rejeições sobre {len(PERIODOS)} snapshots, "
      f"de {uteis} colunas hoje `util`:\n")
if not rejeitadas.empty:
    print(rejeitadas.sort_values(["motivo", "tabela"]).to_string(index=False))
else:
    print("Nenhuma — o filtro já foi aplicado e o doc está em dia.")

> **Veredito 5 — filtro empírico. Uma rejeição.**
>
> `rlEstabUnidAcolhim.tp_sus_nao_sus`, constante em toda a série, reclassificada
> para `descartada`. Colunas `util`: 389 para 388.
>
> O resultado importa mais pelo que **não** mudou: ampliar de duas para nove
> competências acrescentou uma única rejeição. O crivo de D-06 é estável, não
> aperta indefinidamente conforme se olha mais dado.
>
> Rodar esta célula de novo agora deve devolver zero rejeições, porque o doc já
> foi corrigido. Se voltar a apontar algo, é sinal de que o doc e os dados
> divergiram.
>
> **Achado colateral, virou D-20.** A leitura conjunta dos nove snapshots falhou
> na primeira tentativa: três das 44 tabelas têm colunas que somem e voltam, e o
> padrão aponta 201901 como competência anômala em vez de evolução progressiva do
> schema.

## Decisão final

Cinco vereditos fechados. Registrado em `docs/03-decisoes.md`, D-18 a D-22.

| Decisão | Status | Valor fixado | Evidência |
|---|---|---|---|
| D-01 alvo | fechada | `rlEstabEquipamento` | 34.571 aquisições; `rlEstabServClass` empata em volume mas é classificação administrativa |
| D-04 / D-10 densidade | fechada | nove snapshots anuais | taxa entre 0,082 e 0,112, série plana |
| D-06 filtro empírico | fechada | uma rejeição a mais | crivo estável de 2 para 9 competências |
| Chave natural | fechada | as duas hipóteses confirmadas | zero duplicatas em todos os snapshots |
| Trilha 3 geográfica | fechada | 85,7% no estado | D-22 corrige o 57% de D-17 |
| D-21 recorte | fechada | estado de São Paulo | 2,9x eventos, 645 municípios |

**Achados que não estavam previstos:**

1. **A prevalência é severa e irredutível** — 0,047% no estado. Duas restrições
   do espaço de candidatos foram testadas e rejeitadas: por par (tipo de unidade,
   equipamento) corta só 1,5%; por estabelecimento já equipado corta 50% dos
   candidatos mas leva 33% dos positivos e excluiria a primeira aquisição, o
   evento mais relevante para política pública. **MAP@k** passa a ser a métrica
   de destaque. D-19.
2. **Trocar o recorte inverteu a liderança entre alvos.** Uma conclusão medida no
   município não sobreviveu à mudança de escala — lição de método além do
   resultado.
3. **A modelagem do grafo relacional estava errada.** Um nó por linha de tabela
   de fato dá 76 milhões de nós no estado e, pior, não cria vizinho compartilhado
   entre estabelecimentos com o mesmo equipamento. As tabelas do CNES são listas
   de arestas. D-25.

**Consequências aplicadas ao código:**

- `docs/01-selecao-tabelas.md` — `rlEstabUnidAcolhim.tp_sus_nao_sus` descartada
- `docs/02-metodologia.md` — MAP@k como métrica de destaque; recorte estadual
- `src/graph.py` — `RECORTE_PADRAO = '35'`, recorte como prefixo IBGE
- `src/gnn.py` — grafo por categoria, e corte anterior a todos os rótulos (D-25)
- `src/tasks.py` — amostragem de negativos só no treino (D-23)